## Introduction: What Are Profilers and Why Do We Use Them?

When working with deep learning models, especially large neural networks, **performance matters**. Even if a model is mathematically correct, it may be too slow or memory-hungry to be practical. This is where **profilers** become essential tools.

A **profiler** is a tool that measures how a program executes. Instead of telling us *what* the program does, it tells us **how long it takes**, **which parts are expensive**, and **how resources are used** during execution.

---

### What Can a Profiler Measure?

Depending on the configuration, a profiler can provide information about:

* **Execution time**

  * per function
  * per operator
  * per hardware device (CPU, GPU, XPU)
* **Call counts**

  * how many times a function or operator is executed
* **Memory usage**

  * how much memory is allocated and freed
* **Tensor shapes**

  * what input sizes are passed to individual operators
* **Hardware kernels**

  * low-level operations executed on accelerators (e.g. GPU kernels)

---

### Why Profiling Is Important in Deep Learning

In deep learning frameworks such as PyTorch, a single line of Python code can trigger **dozens or hundreds of low-level operations**. Without profiling:

* Performance bottlenecks are **hard to identify**
* Optimizations are often based on **guesswork**
* Slowdowns caused by input size or hardware placement may go unnoticed

Profiling allows us to **replace assumptions with measurements**.

---

### How Profilers Are Usually Used

Profilers are typically applied in **three steps**:

1. **Define a profiling scope**
   Select the part of code you want to analyze (e.g. model inference or training step).

2. **Run the code under the profiler**
   Execute the program while collecting timing and resource data.

3. **Analyze the results**
   Inspect tables or traces to identify expensive operations and patterns.

In PyTorch, profiling is usually done using a **context manager**, which ensures that only the selected code region is measured.

# Profiling a ResNet Model in PyTorch

## Instantiating a Simple ResNet Model

We begin by creating a standard **ResNet-18** model and preparing a batch of synthetic input data.

In [2]:
from torch.profiler import profile, ProfilerActivity, record_function
import torch
from torchvision import models

model = models.resnet18()
inputs = torch.randn(5, 3, 224, 224)

### Explanation

* `models.resnet18()` creates a **ResNet-18** architecture with randomly initialized weights.
* `torch.randn(5, 3, 224, 224)` simulates a batch of:

  * `5` images
  * `3` color channels (RGB)
  * spatial resolution `224 Ã— 224`

This shape matches the expected input format for ResNet models trained on ImageNet.

### Interpretation

At this stage:

* We are **not training** the model.
* We are performing a **forward pass only**, which is ideal for profiling inference performance.

---

## Using the Profiler to Analyze Execution Time

PyTorch provides a powerful **profiler** that allows us to measure:

* Execution time of operators
* Call counts
* Input tensor shapes
* CPU, GPU (CUDA), or XPU activity

The profiler is activated using a **context manager**.

---

### Key Profiler Parameters

```python
activities = [
    ProfilerActivity.CPU,   # CPU-side PyTorch operators
    ProfilerActivity.CUDA,  # GPU kernels (if available)
    ProfilerActivity.XPU    # XPU kernels (if available)
]
```

Other important options:

* `record_shapes=True` â€“ records input tensor shapes for each operator
* `profile_memory=True` â€“ tracks tensor memory usage

---

## Profiling CPU Execution

In [3]:
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("model_inference"):
        model(inputs)
    with record_function("model_inference_again"):
        model(inputs)

### Explanation

* `profile(...)` defines the profiling scope.
* `record_function("model_inference")` assigns a **human-readable label** to this code region.
* Only operations executed inside this context are measured.

### Why use `record_function`?

It allows you to:

* Separate model inference from data loading or preprocessing
* Clearly identify high-level stages in the profiling output

---

## Inspecting the Profiling Results

In [4]:
print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=10
))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     aten::conv2d         0.13%     179.671us        67.55%      95.229ms       2.381ms            40  
                aten::convolution         0.35%     497.082us        67.42%      95.050ms       2.376ms            40  
               aten::_convolution         0.23%     329.964us        67.07%      94.553ms       2.364ms            40  
                aten::thnn_conv2d         0.09%     126.452us        66.82%      94.198ms       2.355ms            40  
       aten::_slow_conv2d_forward        66.46%      93.683ms        66.73%      94.072ms       2.352ms            40  
                  model_inference       

### Interpretation

#### Self CPU Time vs CPU Total Time

* **Self CPU time**: time spent *only* in that operator
* **CPU total time**: includes time spent in all child operators

---

## Grouping by Input Shapes

For deeper insight, we can group operators by their input tensor shapes.

In [4]:
print(
    prof.key_averages(group_by_input_shape=True)
        .table(sort_by="cpu_time_total", row_limit=10)
)

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls                                                                      Input Shapes  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                  model_inference         5.10%       2.669ms       100.00%      52.367ms      52.367ms             1                                                                                []  
                     aten::conv2d         0.03%      15.668us        16.06%       8.411ms       2.103ms             4                             [[5, 64, 56, 56], [64, 64, 3, 3], [], [], [], 

### Why This Matters

* The same operator (e.g. `aten::conv2d`) can be called with **different tensor sizes**
* Larger feature maps typically result in higher execution time

### Example Interpretation

```text
aten::conv2d  [5, 64, 56, 56] â†’ expensive early-layer convolution
aten::conv2d  [5, 512, 7, 7] â†’ deeper-layer convolution
```

This reflects the **hierarchical structure of ResNet**, where spatial resolution decreases and channel depth increases.

---

## Profiling on Accelerators (CUDA / XPU)

PyTorch allows seamless profiling across devices.

In [ ]:
import sys

activities = [ProfilerActivity.CPU]

if torch.cuda.is_available():
    device = "cuda"
    activities.append(ProfilerActivity.CUDA)
elif torch.backends.mps.is_available():
    device = "mps"
elif torch.xpu.is_available():
    device = "xpu"
    activities.append(ProfilerActivity.XPU)
else:
    device = "cpu"


### Running the Model on the Selected Device

In [6]:
model = models.resnet18().to(device)
inputs = torch.randn(5, 3, 224, 224).to(device)

sort_by_keyword = device + "_time_total" if device != "mps" and device != "cpu" else "cpu_time_total"

with profile(activities=activities, record_shapes=True) as prof:
    with record_function("model_inference"):
        model(inputs)

print(prof.key_averages().table(sort_by=sort_by_keyword, row_limit=10))


--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                 model_inference         0.83%       1.125ms       100.00%     135.239ms     135.239ms             1  
                    aten::conv2d         0.03%      41.166us        58.45%      79.053ms       3.953ms            20  
               aten::convolution         0.07%      99.164us        58.42%      79.012ms       3.951ms            20  
              aten::_convolution         0.06%      78.459us        58.35%      78.913ms       3.946ms            20  
          aten::_mps_convolution        58.23%      78.744ms        58.29%      78.834ms       3.942ms            20  
                aten::batch_norm         0.03%  

## Interpreting CUDA Profiling Results

Example (simplified):

| Name              | Self CUDA | CUDA total |
| ----------------- | --------- | ---------- |
| model_inference   | 0 Âµs      | 11.7 ms    |
| aten::conv2d      | 0 Âµs      | 10.5 ms    |
| sgemm_32x32x32_NN | 3.2 ms    | 3.2 ms     |

### Interpretation

* Most high-level PyTorch ops have **zero self CUDA time**
* Actual work happens in **low-level GPU kernels**, such as:

  * `sgemm_*` â†’ matrix multiplication
  * `im2col_kernel` â†’ convolution preparation

This highlights how **PyTorch operators map to optimized hardware kernels**.

## Using the Profiler to Analyze Memory Consumption

In addition to execution time, the profiler in PyTorch can also measure **memory usage** during model execution. This is particularly important for deep learning models, where memory consumption often limits batch size and scalability.

The profiler can report:

* how much memory is **allocated**
* how much memory is **released**
* which operators are responsible for memory usage

To enable memory profiling, we must pass `profile_memory=True`.

---

### Example: Profiling Memory Usage on CPU

In [7]:
model = models.resnet18()
inputs = torch.randn(5, 3, 224, 224)

with profile(
    activities=[ProfilerActivity.CPU],
    profile_memory=True,
    record_shapes=True
) as prof:
    model(inputs)

print(
    prof.key_averages()
        .table(sort_by="self_cpu_memory_usage", row_limit=10)
)

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                      aten::empty         0.45%     198.791us         0.45%     198.791us       0.994us     310.41 MB     310.41 MB           200  
                    aten::resize_         0.11%      48.456us         0.11%      48.456us       2.423us      42.11 MB      42.11 MB            20  
    aten::max_pool2d_with_indices         7.33%       3.230ms         7.33%       3.230ms       3.230ms      11.48 MB      11.48 MB             1  
                      aten::addmm         0.54%     239.875us         0.55%     244.625us     244.625us      19.

### Example Output (Simplified)

| Name                          | CPU Mem  | Self CPU Mem | # Calls |
| ----------------------------- | -------- | ------------ | ------- |
| aten::empty                   | 94.79 MB | 94.79 MB     | 121     |
| aten::max_pool2d_with_indices | 11.48 MB | 11.48 MB     | 1       |
| aten::addmm                   | 19.53 KB | 19.53 KB     | 1       |

---

### Interpretation

* **CPU Mem** â€“ total memory associated with this operator (including child calls)
* **Self CPU Mem** â€“ memory allocated or released *directly* by this operator
* `aten::empty` dominates memory allocation:

  * This operator is used internally to allocate tensors
  * It appears many times during model execution

Memory allocation is **not evenly distributed** across operators. A small number of operators are responsible for most of the memory usage.

---

### Total Memory Usage by Operator

We can also sort by total memory usage instead of self memory:

In [8]:
print(
    prof.key_averages()
        .table(sort_by="cpu_memory_usage", row_limit=10)
)

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                      aten::empty         0.45%     198.791us         0.45%     198.791us       0.994us     310.41 MB     310.41 MB           200  
                 aten::batch_norm         0.09%      41.581us        18.03%       7.947ms     397.344us      47.41 MB           0 B            20  
     aten::_batch_norm_impl_index         0.22%      95.918us        17.93%       7.905ms     395.265us      47.41 MB           0 B            20  
          aten::native_batch_norm        17.30%       7.624ms        17.68%       7.793ms     389.635us      47.

### Example Output (Simplified)

| Name             | CPU Mem  | Self CPU Mem | # Calls |
| ---------------- | -------- | ------------ | ------- |
| aten::batch_norm | 47.41 MB | 0 B          | 20      |
| aten::conv2d     | 47.37 MB | 0 B          | 20      |
| aten::max_pool2d | 11.48 MB | 0 B          | 1       |

---

### Interpretation

* Many operators show **zero self memory usage**
* They rely on **child operators** (such as `aten::empty`) to allocate memory
* This reflects how PyTorch builds computations from **composable primitives**